In [ ]:
import os
import numpy as np
import imageio.v2 as imageio
from PIL import Image

# --------------------------------------------------
# CONFIGURACION
# --------------------------------------------------
nombre_archivo = "foto.jpeg"
nombre_salida = "image.mif"

if not os.path.exists(nombre_archivo):
    print(f"ERROR: No se encontró '{nombre_archivo}'")
else:
    # 1. Cargar imagen
    imagen_raw = imageio.imread(nombre_archivo)

    # 2. Asegurar RGB
    im_pil = Image.fromarray(imagen_raw).convert("RGB")

    # 3. Redimensionar a 640x480
    im_resized = im_pil.resize((640, 480))
    imagen_final = np.asarray(im_resized, dtype=np.uint8)

    # 4. Convertir a RGB332
    #    R: 3 bits
    #    G: 3 bits
    #    B: 2 bits
    red   = imagen_final[:, :, 0]
    green = imagen_final[:, :, 1]
    blue  = imagen_final[:, :, 2]

    rgb332 = ((red >> 5) << 5) | ((green >> 5) << 2) | (blue >> 6)
    datos = rgb332.flatten()

    # 5. Generar .mif
    depth = 640 * 480
    width = 8

    with open(nombre_salida, "w") as f:
        f.write(f"DEPTH = {depth};\n")
        f.write(f"WIDTH = {width};\n")
        f.write("ADDRESS_RADIX = HEX;\n")
        f.write("DATA_RADIX = HEX;\n")
        f.write("CONTENT BEGIN\n")

        for i in range(depth):
            f.write(f"{i:05X} : {int(datos[i]):02X};\n")

        f.write("END;\n")

    print("--------------------------------------------------")
    print(f"¡Éxito! Archivo '{nombre_salida}' generado.")
    print("Este MIF ahora está en RGB332 de 8 bits.")
    print("--------------------------------------------------")